# Gold · indicadores_mensais

**Origem:** `gold.chamados_nota`, `silver.custos` e `silver.canais` → **Destino:** `vcredit_gold.indicadores_mensais`
**Grão:** uma linha por mês e canal · **Depende de:** `gold_chamados_nota` e as silvers

Principais indicadores da operação, calculados com os dados tratados, para o painel: volume, custo, tempos reais, resolução, abandono e satisfação.

| Indicador | Como |
|---|---|
| `custo_total`, `custo_medio` | Um custo por chamado (silver.custos); chamados sem custo ficam fora |
| `espera_media_min`, `atend_medio_min` | Tempos reais (`espera_seg` e `atend_seg` da chamados_nota) |
| `pct_resolvido`, `pct_abandono` | Sobre o total de chamados do mês e canal |
| `nota_media`, `taxa_resposta_pct` | Só chamados com nota; taxa = chamados com nota ÷ total |

**Decisões:** os 199 chamados sem data ficam fora (não têm mês). Pipeline idempotente (sobrescreve a tabela).

In [0]:
from pyspark.sql import functions as F

cat = "rocket_lab"
s, g = f"{cat}.vcredit_silver", f"{cat}.vcredit_gold"

# um custo por chamado
cu = (spark.table(f"{s}.custos").filter("custo is not null")
        .groupBy("id_chamado").agg(F.max("custo").alias("custo")))

ca = spark.table(f"{s}.canais").select(F.col("nome_canal").alias("canal"), "tipo_canal")


base = (spark.table(f"{g}.chamados_nota")
          .filter("mes is not null")
          .join(cu, "id_chamado", "left")
          .join(ca, "canal", "left"))

ind = (base.groupBy("mes", "canal", "tipo_canal").agg(
    F.count("*").alias("chamados"),
    F.round(F.sum("custo"), 2).alias("custo_total"),
    F.round(F.avg("custo"), 2).alias("custo_medio"),
    F.round(F.avg((F.col("resolvido") == "sim").cast("int")) * 100, 1).alias("pct_resolvido"),
    F.round(F.avg(F.col("abandono").cast("int")) * 100, 1).alias("pct_abandono"),
    F.round(F.avg("espera_seg") / 60, 1).alias("espera_media_min"),
    F.round(F.avg("atend_seg") / 60, 1).alias("atend_medio_min"),
    F.round(F.avg("nota"), 2).alias("nota_media"),
    F.round(F.avg(F.col("tem_nota").cast("int")) * 100, 1).alias("taxa_resposta_pct")))

(ind.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{g}.indicadores_mensais"))

display(spark.table(f"{g}.indicadores_mensais").orderBy("mes", "canal"))